## Phase 3: Heuristic Hybrid Re-ranking

Two-stage retrieval: FAISS surfaces a broad semantic candidate pool (top 50), then a
manual heuristic combines semantic similarity, rating, and popularity.

This is a non-learned baseline. The actual Learning-to-Rank model belongs in the next
notebook (NB 9).


In [1]:
import os
import json
import math
import pickle
import string

import pandas as pd
import numpy as np
import faiss

from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from collections import Counter


/home/tommaso/projects/Board-Game-Data-Analytics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cached artifacts produced upstream

NB8 is evaluation-only: it consumes the artifacts generated by NB6 and NB7. It never creates or overwrites them.

- `data/embeddings_nat.npy` — natural-text document embeddings from NB7
- `data/faiss_index.bin` — FAISS index built by NB7
- `data/tokens.pkl` — BM25 tokens from `Description_nat`, generated by NB6

If one of these artifacts is missing or has the wrong number of rows, NB8 stops with an error. Regeneration belongs to NB6/NB7.


### Re-ranking function

Weights (0.5 semantic / 0.3 rating / 0.2 popularity) define a manual heuristic baseline.
They are not learned from qrels. The actual supervised Learning-to-Rank experiment belongs
to NB 9.


In [2]:
DATA_DIR = '../data'
EMBEDDINGS_PATH = os.path.join(DATA_DIR, 'embeddings_nat.npy')
FAISS_INDEX_PATH = os.path.join(DATA_DIR, 'faiss_index.bin')
TOKENS_PATH = os.path.join(DATA_DIR, 'tokens.pkl')
DETAILS_PATH = os.path.join(DATA_DIR, 'details.csv')

# Query encoder only. Document embeddings are loaded from NB7; they are never regenerated here.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Rebuild the dataframe from the same source rows used upstream.
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]

df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)

# Natural-language description: same source used by NB6/NB7/qrels v4.
if not os.path.exists(DETAILS_PATH):
    raise FileNotFoundError(
        f"{DETAILS_PATH} not found. NB8 requires details.csv to reconstruct Description_nat."
    )

orig = pd.read_csv(DETAILS_PATH).drop_duplicates('id')
m = df[['BGGId']].merge(
    orig[['id', 'description']],
    left_on='BGGId', right_on='id', how='left'
)
natural_text = (
    m['description']
    .fillna('')
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
df['Description_nat'] = natural_text.where(
    natural_text.str.len() > 0, df['Description'].astype(str)
)

print(f"Dataset ready! Total games: {len(df)}")

# ------------------------------------------------------------------
# LOAD ONLY: document embeddings from NB7.
if not os.path.exists(EMBEDDINGS_PATH):
    raise FileNotFoundError(
        f"{EMBEDDINGS_PATH} not found. Run NB7 first. NB8 does not generate embeddings."
    )

embeddings = np.load(EMBEDDINGS_PATH).astype('float32')
if embeddings.shape[0] != len(df):
    raise ValueError(
        f"Embedding cache row count mismatch ({embeddings.shape[0]} vs {len(df)}). "
        "Regenerate embeddings_nat.npy by running NB7."
    )
print(f"Natural-text embeddings loaded: {embeddings.shape}")

# LOAD ONLY: FAISS index from NB7.
if not os.path.exists(FAISS_INDEX_PATH):
    raise FileNotFoundError(
        f"{FAISS_INDEX_PATH} not found. Run NB7 first. NB8 does not build FAISS."
    )

index = faiss.read_index(FAISS_INDEX_PATH)
if index.ntotal != len(df):
    raise ValueError(
        f"FAISS index has {index.ntotal} vectors, expected {len(df)}. "
        "Regenerate faiss_index.bin by running NB7."
    )
print(f"FAISS index loaded: {index.ntotal} vectors")

print("Setup complete. NB8 is evaluation-only; no document artifact was generated.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11956.53it/s]


Dataset ready! Total games: 19728
Natural-text embeddings loaded: (19728, 384)
FAISS index loaded: 19728 vectors
Setup complete. NB8 is evaluation-only; no document artifact was generated.


In [3]:
# Final artifact integrity check
assert os.path.exists(EMBEDDINGS_PATH)
assert os.path.exists(FAISS_INDEX_PATH)
assert os.path.exists(TOKENS_PATH)
print('NB8 artifact check: embeddings_nat.npy, faiss_index.bin and tokens.pkl are present.')


NB8 artifact check: embeddings_nat.npy, faiss_index.bin and tokens.pkl are present.


In [4]:
def search_and_rerank_heuristic(query, top_candidates=50, final_top_k=5):
    """Retrieve candidates with FAISS and rerank them with a manual heuristic."""
    query_vector = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, top_candidates)

    candidates = df.iloc[indices[0]].copy()
    candidates['faiss_distance'] = distances[0]

    scaler = MinMaxScaler()

    # Higher semantic similarity is better; FAISS distance is lower-is-better.
    candidates['semantic_score'] = 1 - scaler.fit_transform(
        candidates[['faiss_distance']]
    ).ravel()

    candidates['norm_rating'] = scaler.fit_transform(
        candidates[['Rating']]
    ).ravel()

    candidates['norm_popularity'] = scaler.fit_transform(
        candidates[['Avg_popularity_log']]
    ).ravel()

    # Manual heuristic baseline; this is NOT learned.
    W_SEMANTIC = 0.50
    W_RATING = 0.30
    W_POPULARITY = 0.20

    candidates['heuristic_score'] = (
        candidates['semantic_score'] * W_SEMANTIC
        + candidates['norm_rating'] * W_RATING
        + candidates['norm_popularity'] * W_POPULARITY
    )

    return candidates.sort_values(
        by='heuristic_score', ascending=False
    ).head(final_top_k)[
        ['BGGId', 'Name', 'heuristic_score', 'semantic_score',
         'norm_rating', 'Rating', 'Description_nat']
    ]


### Test the LTR engine
Same query used throughout notebooks 6-8, so results are directly comparable across BM25,
pure FAISS, and this re-ranked version.

In [5]:
user_query = "A cooperative space game involving betrayal"
print(f"Heuristic Hybrid Results for: '{user_query}'\n")

heuristic_results = search_and_rerank_heuristic(
    user_query, top_candidates=50, final_top_k=5
)

for _, row in heuristic_results.iterrows():
    print(f"{row['Name']} (BGGId={int(row['BGGId'])})")
    print(
        f"[Heuristic: {row['heuristic_score']:.2f} | "
        f"Semantic: {row['semantic_score']:.2f} | "
        f"Rating: {row['Rating']:.1f}]"
    )
    print(f"Snippet: {row['Description_nat'][:150]}...\n")


Heuristic Hybrid Results for: 'A cooperative space game involving betrayal'

Cosmic Encounter (BGGId=39463)
[Heuristic: 1.00 | Semantic: 1.00 | Rating: 3.0]
Snippet: Build a galactic empire...&#10;&#10;In the depths of space, the alien races of the Cosmos vie with each other for control of the universe. Alliances f...

The Crew: The Quest for Planet Nine (BGGId=284083)
[Heuristic: 0.79 | Semantic: 0.59 | Rating: 3.0]
Snippet: In the co-operative trick-taking game The Crew: The Quest for Planet Nine, the players set out as astronauts on an uncertain space adventure. What are...

Sabotage (BGGId=252854)
[Heuristic: 0.55 | Semantic: 0.27 | Rating: 3.0]
Snippet: Sabotage is a game of team stealth tactics. Two teams compete, with spies trying to save the world and the villains trying to stop them. Spies must be...

Battlestar Galactica: The Board Game (BGGId=37111)
[Heuristic: 0.50 | Semantic: 0.02 | Rating: 3.0]
Snippet: Battlestar Galactica: The Board Game is an exciting game of mistrust,

## Phase 4: System Evaluation (Effectiveness Measures)

Three hand-labeled queries, used as a first sanity check on the metric implementations before
scaling up to an LLM-as-a-Judge over 100 queries in Phase 4.1 - small enough to eyeball by hand,
big enough to catch an obviously broken MRR/NDCG calculation.

Relevance scale: 3 = perfect match, 2 = highly relevant, 1 = partially relevant, 0 = irrelevant.


In [6]:
# Ground Truth dictionary (Qrels)
qrels = {
    "A cooperative space game involving betrayal": {
        "Battlestar Galactica: The Board Game": 3,
        "Nemesis": 3,
        "Unfathomable": 2, # Betrayal, but not space (it's ocean/Cthulhu)
        "The Crew: The Quest for Planet Nine": 1, # Space and coop, but no betrayal
        "Catan": 0
    },
    "Fast paced card game for two players": {
        "7 Wonders Duel": 3,
        "Jaipur": 3,
        "Lost Cities": 3,
        "Dominion": 2, # Card game, supports 2 players, but maybe not purely "fast paced 2p only"
        "Twilight Imperium": 0 # Exact opposite
    },
    "Heavy economic eurogame with route building": {
        "Brass: Birmingham": 3,
        "Brass: Lancashire": 3,
        "Great Western Trail": 3,
        "Ticket to Ride": 1, # Route building, but very light (not heavy economic)
        "Monopoly": 0
    }
}

test_queries = list(qrels.keys())
print(f"Loaded {len(test_queries)} test queries.")

# Convert the small sanity-check fixture to the canonical BGGId representation.
# The evaluation pipeline uses BGGId throughout; names are used here only to keep
# Convert the manually authored sanity-check judgments to BGGId.
# Game names are not globally unique in the corpus, so resolve each name only
# when it maps to exactly one record; otherwise use the explicit BGGId fixture.
# The fixture below is intentionally kept separate from the full-corpus qrels.

_name_to_ids = df.groupby('Name')['BGGId'].apply(
    lambda s: [int(x) for x in s.tolist()]
).to_dict()

# Keep only names that are unambiguous in the corpus.
# Ambiguous names are omitted from this tiny sanity check rather than
# arbitrarily selecting one of several games with the same title.
_ambiguous = {
    name: ids for name, ids in _name_to_ids.items() if len(ids) != 1
}
if _ambiguous:
    print(
        f"Sanity-check warning: {len(_ambiguous)} names are duplicated in the "
        "full corpus; ambiguous manual examples will be skipped."
    )

qrels = {
    query: {
        _name_to_ids[name][0]: score
        for name, score in judgments.items()
        if name in _name_to_ids and len(_name_to_ids[name]) == 1
    }
    for query, judgments in qrels.items()
}


Loaded 3 test queries.
Sanity-check warning: 298 names are duplicated in the full corpus; ambiguous manual examples will be skipped.


## Metric Calculation Functions

MRR and NDCG@5 use the same BGGId-based qrels and exponential gain for graded relevance.


In [7]:
def calculate_mrr(retrieved_docs, query_qrels):
    """Reciprocal rank of the first retrieved document with positive relevance."""
    for rank, doc_id in enumerate(retrieved_docs, start=1):
        if query_qrels.get(int(doc_id), 0) > 0:
            return 1.0 / rank
    return 0.0


def calculate_ndcg(retrieved_docs, query_qrels, k=5):
    """NDCG@k for graded relevance judgments using exponential gain."""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_docs[:k]):
        rel_score = query_qrels.get(int(doc_id), 0)
        dcg += (2 ** rel_score - 1) / math.log2(i + 2)

    ideal_scores = sorted(query_qrels.values(), reverse=True)
    idcg = sum(
        (2 ** rel - 1) / math.log2(i + 2)
        for i, rel in enumerate(ideal_scores[:k])
    )

    return 0.0 if idcg == 0.0 else dcg / idcg


## Model Comparison: BM25 vs Heuristic Hybrid Re-ranking

Checks whether the FAISS + heuristic re-ranking pipeline actually beats plain lexical search
(BM25) on this small labeled set, before scaling up to the expanded evaluation set.


In [8]:
TOKENS_PATH = os.path.join(DATA_DIR, 'tokens.pkl')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return [
        word for word in word_tokenize(text)
        if word not in stop_words
    ]

# LOAD ONLY: BM25 tokens generated by NB6 from Description_nat.
if not os.path.exists(TOKENS_PATH):
    raise FileNotFoundError(
        f"{TOKENS_PATH} not found. Run NB6 first. NB8 does not generate BM25 tokens."
    )

with open(TOKENS_PATH, 'rb') as f:
    tokens_list = pickle.load(f)

if len(tokens_list) != len(df):
    raise ValueError(
        f"tokens.pkl has {len(tokens_list)} entries, expected {len(df)}. "
        "Regenerate tokens.pkl by running NB6."
    )

df['tokens'] = tokens_list
bm25 = BM25Okapi(df['tokens'].tolist())

def search_games_bm25(query, top_k=5):
    query_tokens = preprocess_text(query)
    doc_scores = bm25.get_scores(query_tokens)

    results = df[['BGGId', 'Name', 'Description_nat']].copy()
    results['bm25_score'] = doc_scores
    return results.sort_values(
        by='bm25_score', ascending=False
    ).head(top_k)

print("BM25 loaded from NB6 artifact; no tokenization/cache generation performed.")


BM25 loaded from NB6 artifact; no tokenization/cache generation performed.


In [9]:
bm25_mrr_total = 0
bm25_ndcg_total = 0
hybrid_mrr_total = 0
hybrid_ndcg_total = 0

print("Starting sanity-check evaluation...\n")

for query in test_queries:
    query_qrels = qrels[query]

    bm25_results = search_games_bm25(query, top_k=5)['BGGId'].astype(int).tolist()
    bm25_mrr = calculate_mrr(bm25_results, query_qrels)
    bm25_ndcg = calculate_ndcg(bm25_results, query_qrels, k=5)

    hybrid_results = search_and_rerank_heuristic(
        query, top_candidates=50, final_top_k=5
    )['BGGId'].astype(int).tolist()
    hybrid_mrr = calculate_mrr(hybrid_results, query_qrels)
    hybrid_ndcg = calculate_ndcg(hybrid_results, query_qrels, k=5)

    bm25_mrr_total += bm25_mrr
    bm25_ndcg_total += bm25_ndcg
    hybrid_mrr_total += hybrid_mrr
    hybrid_ndcg_total += hybrid_ndcg

    print(f"Query: {query}")
    print(f"  BM25     -> MRR: {bm25_mrr:.3f} | NDCG@5: {bm25_ndcg:.3f}")
    print(f"  Heuristic-> MRR: {hybrid_mrr:.3f} | NDCG@5: {hybrid_ndcg:.3f}\n")

num_queries = len(test_queries)

print("FINAL AVERAGE SCORES")
print(f"BM25       - Mean MRR: {bm25_mrr_total / num_queries:.3f} | "
      f"Mean NDCG@5: {bm25_ndcg_total / num_queries:.3f}")
print(f"Heuristic  - Mean MRR: {hybrid_mrr_total / num_queries:.3f} | "
      f"Mean NDCG@5: {hybrid_ndcg_total / num_queries:.3f}")


Starting sanity-check evaluation...

Query: A cooperative space game involving betrayal
  BM25     -> MRR: 0.000 | NDCG@5: 0.000
  Heuristic-> MRR: 0.500 | NDCG@5: 0.273

Query: Fast paced card game for two players
  BM25     -> MRR: 0.000 | NDCG@5: 0.000
  Heuristic-> MRR: 0.000 | NDCG@5: 0.000

Query: Heavy economic eurogame with route building
  BM25     -> MRR: 0.000 | NDCG@5: 0.000
  Heuristic-> MRR: 0.000 | NDCG@5: 0.000

FINAL AVERAGE SCORES
BM25       - Mean MRR: 0.000 | Mean NDCG@5: 0.000
Heuristic  - Mean MRR: 0.167 | Mean NDCG@5: 0.091


## Phase 4.1: Automated Qrel Generation (LLM-as-a-Judge)

Hand-labeling qrels doesn't scale past a handful of queries, so the expanded evaluation set
(100 queries) is generated with an LLM judge (`gemini-3.5-flash-lite` when available).
Candidates are pooled from the top 20 of BM25 and the top 20 of pure FAISS retrieval per query.
The two lists are merged, so the judge evaluates only candidates surfaced by at least one of
the two base retrieval systems.

**Ranking, not scoring.** The model ranks the whole pool for a query rather than assigning a
raw 0-3 score to each game independently; the score is then derived deterministically from rank
position (top 2 -> 3, next 6 -> 2, next 8 -> 1, rest -> 0). Asking for a raw score directly
lets the model's idea of relevance drift between queries; asking it to rank the complete pool
makes the subsequent bucket mapping deterministic and consistent across queries.

These are LLM-generated graded relevance judgments, not human ground truth. They are also
pool-based: games outside the BM25/FAISS candidate pool are not judged.

Every query is generated with the same method and model. A checkpoint built partially with a
different method/model is discarded rather than mixed in.

The API call itself - rate-limiting, retries, JSON parsing - lives in
`scripts/generate_qrels_v4.py` and runs as a separate batch job. It produces
`expanded_qrels_v4.json`; it does not belong inline in a notebook meant to be re-run interactively.


In [10]:
# Load the pre-generated Qrels from data/evaluation/
with open('../data/evaluation/expanded_qrels_v4.json', 'r') as f:
    saved = json.load(f)
full_qrels = saved.get('qrels', saved)

print(f"Successfully loaded {len(full_qrels)} annotated queries!")

Successfully loaded 100 annotated queries!


### Verifying the qrels

The v4 qrels identify documents by `BGGId`, not by game name. We therefore verify that every
judged ID exists in the current dataset and that the deterministic 2/6/8 relevance buckets
are respected.

Expected distribution for a pool of N judged games:
- 2 games scored 3
- 6 games scored 2
- 8 games scored 1
- N - 16 games scored 0

Games omitted by the LLM are not silently converted to zero by the v4 generation script.


In [11]:
known_ids = set(df['BGGId'].astype(int))

N_SCORE_3 = 2
N_SCORE_2 = 6
N_SCORE_1 = 8

qrels_file = os.path.join(DATA_DIR, 'evaluation', 'expanded_qrels_v4.json')

with open(qrels_file, 'r') as f:
    saved = json.load(f)

qrels_v4 = saved['qrels']

print(
    f"Method: {saved.get('_metadata', {}).get('method')} | "
    f"Model: {saved.get('_metadata', {}).get('model')}"
)
print(f"Total queries: {len(qrels_v4)}")
print(f"Total judgments: {sum(len(v) for v in qrels_v4.values())}\n")

# 1. Every qrel document ID must exist in the dataset.
ids_ok = True
for query, scores in qrels_v4.items():
    invalid_ids = [
        int(gid) for gid in scores
        if int(gid) not in known_ids
    ]
    if invalid_ids:
        ids_ok = False
        print(f"Query: '{query}'")
        print(f"  Invalid BGGIds: {invalid_ids}")

if ids_ok:
    print("All judged BGGIds exist in the dataset.\n")

# 2. Verify the deterministic 2/6/8 bucket mapping.
bucket_ok = True
for query, scores in qrels_v4.items():
    dist = Counter(scores.values())
    n = len(scores)

    expected = {
        3: min(N_SCORE_3, n),
        2: min(N_SCORE_2, max(0, n - N_SCORE_3)),
        1: min(N_SCORE_1, max(0, n - N_SCORE_3 - N_SCORE_2)),
    }
    expected[0] = max(
        0, n - N_SCORE_3 - N_SCORE_2 - N_SCORE_1
    )

    actual = {score: dist.get(score, 0) for score in [3, 2, 1, 0]}

    if actual != expected:
        bucket_ok = False
        print(f"[BUCKET CHECK] '{query}'")
        print(f"  Expected: {expected}")
        print(f"  Actual:   {actual}")

if bucket_ok:
    print("All queries follow the expected 2/6/8 bucket mapping.")


Method: ranking_bucket_expanded_v4_ids | Model: gemini-3.5-flash-lite
Total queries: 100
Total judgments: 3547

All judged BGGIds exist in the dataset.

All queries follow the expected 2/6/8 bucket mapping.


Full listing below, sorted by relevance, for a manual spot-check of the flagged queries (or any
others).

In [12]:
id2name = dict(zip(df['BGGId'].astype(int), df['Name']))

for query, scores in qrels_v4.items():
    print(f"\n{query}")
    for gid, score in sorted(scores.items(), key=lambda x: -x[1]):
        gid = int(gid)
        name = id2name.get(gid, 'UNKNOWN')
        print(f"  {score}  {gid}  {name}")



Cyberpunk game with hacking mechanics
  3  23081  HeroCard Cyberspace
  3  163963  League of Hackers
  2  254123  Shadowrun Crossfire: Prime Runner Edition
  2  253087  Shadowrun: Sprawl Ops
  2  1250  Hacker
  2  10707  Hacker: Deluxe Edition
  2  229319  Human Interface: Be a Better Human
  2  11826  Cyberpunk: The Collectible Card Game
  1  118063  Android: Infiltration
  1  94493  Zaibatsu
  1  233371  Clank! In! Space!: A Deck-Building Adventure
  1  145766  Gamedec
  1  258752  Bitcoin Hackers
  1  155245  In Security
  1  181440  Hack Trick
  1  234860  Hack Trick: It's Hacking Time
  0  201971  Super Hack Override
  0  308126  The Coding
  0  3306  Hack Attack
  0  98960  Rogue Agent
  0  252854  Sabotage
  0  273933  Techlandia
  0  42929  Martian Rails
  0  39  Darkover
  0  196712  Battlestations: Second Edition
  0  8103  .hack//ENEMY
  0  312951  Alien Puppies
  0  173514  Meeple Quest
  0  282237  Heist: One Team, One Mission
  0  33950  Solo Tower Hack
  0  199309  Path

## Phase 4.2: Large-Scale Evaluation (BM25 vs. Heuristic Hybrid)

The same comparison as Phase 4 is repeated over the 100 LLM-judged queries.

The v4 qrels are ID-based (`BGGId`), so retrieval outputs are converted to `BGGId` before
metric calculation. This keeps the evaluation robust to duplicate game names.


In [13]:
# Load the v4 qrels once; no second read of the old qrels file.
qrels_file = os.path.join(DATA_DIR, 'evaluation', 'expanded_qrels_v4.json')

with open(qrels_file, 'r') as f:
    saved = json.load(f)

full_qrels = saved['qrels']
id2name = dict(zip(df['BGGId'].astype(int), df['Name']))

bm25_mrr_total = 0.0
bm25_ndcg_total = 0.0
hybrid_mrr_total = 0.0
hybrid_ndcg_total = 0.0

num_queries = len(full_qrels)

print(
    f"Evaluating BM25 vs. Heuristic Hybrid on "
    f"{num_queries} LLM-judged queries...\n"
)

for query, raw_query_qrels in full_qrels.items():
    # JSON object keys are strings; normalize them to integer BGGIds.
    query_qrels = {
        int(gid): int(score)
        for gid, score in raw_query_qrels.items()
    }

    # --- 1. BM25 ---
    bm25_df = search_games_bm25(query, top_k=5)
    bm25_ids = bm25_df['BGGId'].astype(int).tolist()

    bm25_mrr = calculate_mrr(bm25_ids, query_qrels)
    bm25_ndcg = calculate_ndcg(bm25_ids, query_qrels, k=5)

    # --- 2. Phase 3 heuristic hybrid ---
    hybrid_df = search_and_rerank_heuristic(
        query, top_candidates=50, final_top_k=5
    )
    hybrid_ids = hybrid_df['BGGId'].astype(int).tolist()

    hybrid_mrr = calculate_mrr(hybrid_ids, query_qrels)
    hybrid_ndcg = calculate_ndcg(hybrid_ids, query_qrels, k=5)

    bm25_mrr_total += bm25_mrr
    bm25_ndcg_total += bm25_ndcg
    hybrid_mrr_total += hybrid_mrr
    hybrid_ndcg_total += hybrid_ndcg

print(f"FINAL AVERAGE RESULTS ({num_queries} QUERIES)")
print()
print("BM25 Baseline:")
print(f"  Mean MRR:    {bm25_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {bm25_ndcg_total / num_queries:.3f}")

print()
print("Heuristic Hybrid Re-ranking:")
print(f"  Mean MRR:    {hybrid_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {hybrid_ndcg_total / num_queries:.3f}")


Evaluating BM25 vs. Heuristic Hybrid on 100 LLM-judged queries...

FINAL AVERAGE RESULTS (100 QUERIES)

BM25 Baseline:
  Mean MRR:    0.817
  Mean NDCG@5: 0.397

Heuristic Hybrid Re-ranking:
  Mean MRR:    0.827
  Mean NDCG@5: 0.374
